# Lab 1 — Trustworthy RAG: from a UNESCO PDF to cited evidence

## Scenario

A school wants a small assistant that helps staff inspect UNESCO guidance about generative AI in education. The assistant must answer only from an approved document, show where its evidence came from, and say when the document does not support an answer.

In this 30–45 minute English-first lab, you will ingest the full 48-page UNESCO PDF, create multilingual embeddings with multilingual-e5-small, compare lexical and semantic retrieval, and make a bounded response.

**Learning outcomes**

- Explain how RAG differs from retraining a model.
- Ingest a PDF into page records and provenance-preserving chunks.
- Retrieve evidence with normalized multilingual embeddings and cosine similarity.
- Inspect results before making a cited answer or a safe non-answer.

## The Lab 1 RAG model

Approved PDF → page records → chunks + metadata → embeddings/index → retrieve evidence → cited response

RAG changes the **evidence available at question time**. It does not retrain the embedding model or make a language model more knowledgeable. A high similarity score ranks likely-relevant chunks; it does not prove that a claim is true, complete, or fair. Grounding means that a reader can inspect the same retrieved evidence that the assistant used.

### Trust boundary

This lab uses one approved UNESCO source. Every chunk keeps its source page, URL, and licence. The workflow does not search the web, call an answer-generation API, or silently substitute another corpus. If evidence is weak or the question is too broad, the assistant asks for clarification or says that the corpus does not support the request.

In [ ]:
# Run this once in a fresh Google Colab runtime. No API key is required.
%pip -q install pypdf sentence-transformers scikit-learn matplotlib

import math
import re
from pprint import pprint
from typing import Literal, TypedDict

import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import TSNE

## 1. Define the records that travel through the pipeline

A trustworthy RAG system preserves information needed to audit an answer. PageRecord is the extracted source; ChunkRecord is the retrievable evidence; EmbeddingIndex connects chunks to vectors; RetrievalCandidate carries a score; and Citation is the user-visible locator.

In [ ]:
class PageRecord(TypedDict):
    source_id: str
    title: str
    source_url: str
    license: str
    source_file: str
    page: int
    text: str

class ChunkRecord(TypedDict):
    chunk_id: str
    source_id: str
    title: str
    source_url: str
    license: str
    source_file: str
    page_start: int
    page_end: int
    text: str

class EmbeddingIndex(TypedDict):
    model_name: str
    chunk_ids: list[str]
    vectors: np.ndarray

class RetrievalCandidate(TypedDict):
    method: str
    score: float
    chunk: ChunkRecord

class Citation(TypedDict):
    title: str
    chunk_id: str
    source_page: str
    source_url: str
    score: float

class SafeAnswer(TypedDict):
    status: Literal['grounded', 'ambiguous', 'not_found']
    answer: str
    citations: list[Citation]

## 2. Ingest the approved source

Upload the included UNESCO PDF from data/corpus/source-pdfs when Colab opens the file picker. The pipeline extracts text page by page rather than treating the PDF as one string, so citations can point readers back to a page. Pages with little extractable text are reported instead of silently treated as evidence.

In [ ]:
from google.colab import files
from pypdf import PdfReader

SOURCE_ID = 'unesco-genai-guidance-2023'
SOURCE_TITLE = 'Guidance for generative AI in education and research'
SOURCE_URL = 'https://unesdoc.unesco.org/ark:/48223/pf0000386693'
SOURCE_LICENSE = 'CC BY-SA 3.0 IGO'
MIN_PAGE_CHARACTERS = 120

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Upload the approved UNESCO PDF to continue.')
pdf_path = next(iter(uploaded))
reader = PdfReader(pdf_path)
page_records: list[PageRecord] = []
short_or_empty_pages: list[int] = []
for page_number, page in enumerate(reader.pages, start=1):
    page_text = re.sub(r'\s+', ' ', page.extract_text() or '').strip()
    if len(page_text) < MIN_PAGE_CHARACTERS:
        short_or_empty_pages.append(page_number)
        continue
    page_records.append({'source_id': SOURCE_ID, 'title': SOURCE_TITLE, 'source_url': SOURCE_URL, 'license': SOURCE_LICENSE, 'source_file': pdf_path, 'page': page_number, 'text': page_text})

ingestion_report = {'total_pdf_pages': len(reader.pages), 'usable_text_pages': len(page_records), 'short_or_empty_pages': short_or_empty_pages, 'source_file': pdf_path}
pprint(ingestion_report)
print('\nFirst usable page preview:\n', page_records[0]['text'][:600])
if len(page_records) != 46:
    print(f'Warning: expected about 46 usable pages, found {len(page_records)}. Inspect the report before continuing.')

## 3. Chunk evidence without losing provenance

Embeddings work on bounded passages, not an entire book. This lab uses 180-word chunks with a 30-word overlap. Larger chunks retain more context but can blur topics; smaller chunks are more precise but can separate a claim from its qualification. The overlap reduces abrupt context loss at a boundary.

Chunks stay within a page in this introductory pipeline. This makes the citation locator unambiguous and keeps the page-to-chunk relationship easy to inspect.

In [ ]:
CHUNK_WORDS, OVERLAP_WORDS = 180, 30

def chunk_page(page: PageRecord, max_words: int = CHUNK_WORDS, overlap: int = OVERLAP_WORDS) -> list[ChunkRecord]:
    if overlap >= max_words:
        raise ValueError('overlap must be smaller than max_words')
    words, step, records = page['text'].split(), max_words - overlap, []
    for chunk_number, start in enumerate(range(0, len(words), step), start=1):
        chunk_words = words[start:start + max_words]
        if chunk_words:
            records.append({'chunk_id': f"{page['source_id']}-p{page['page']:02d}-c{chunk_number:02d}", 'source_id': page['source_id'], 'title': page['title'], 'source_url': page['source_url'], 'license': page['license'], 'source_file': page['source_file'], 'page_start': page['page'], 'page_end': page['page'], 'text': ' '.join(chunk_words)})
    return records

chunks = [chunk for page in page_records for chunk in chunk_page(page)]
word_counts = [len(chunk['text'].split()) for chunk in chunks]
assert chunks, 'No chunks were created; inspect PDF extraction.'
assert len({chunk['chunk_id'] for chunk in chunks}) == len(chunks), 'Chunk IDs must be unique.'
assert all(chunk['text'].strip() and chunk['source_url'] == SOURCE_URL and chunk['page_start'] for chunk in chunks), 'Content or provenance was lost.'
assert all(count <= CHUNK_WORDS for count in word_counts), 'Chunk exceeds word limit.'
pprint({'chunk_count': len(chunks), 'min_words': min(word_counts), 'median_words': int(np.median(word_counts)), 'max_words': max(word_counts), 'overlap_words': OVERLAP_WORDS})
pprint({key: chunks[0][key] for key in ('chunk_id', 'page_start', 'source_url', 'license')})
print('\nFirst chunk preview:\n', chunks[0]['text'][:700])
if len(chunks) != 206:
    print(f'Warning: expected about 206 chunks with this PDF and configuration, found {len(chunks)}.')

## 4. Build the mandatory multilingual embedding index

multilingual-e5-small maps text with related meaning near one another in a high-dimensional vector space. E5 uses different prefixes for stored passages and questions: passage: and query:. We normalize every vector to length 1, so its dot product with another normalized vector is cosine similarity.

The first run downloads the model from Hugging Face. This is a required dependency for this lab, not an offline fallback. If the download fails, restore network access and rerun this cell rather than replacing the model with a different retrieval method.

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'intfloat/multilingual-e5-small'
try:
    embedding_model = SentenceTransformer(MODEL_NAME)
except Exception as exc:
    raise RuntimeError('Could not load multilingual-e5-small. Check Colab network access, then rerun this cell; do not replace it with a fallback index.') from exc

passages = [f"passage: {chunk['text']}" for chunk in chunks]
chunk_vectors = embedding_model.encode(passages, normalize_embeddings=True, show_progress_bar=True, convert_to_numpy=True)
embedding_index: EmbeddingIndex = {'model_name': MODEL_NAME, 'chunk_ids': [chunk['chunk_id'] for chunk in chunks], 'vectors': chunk_vectors}
assert np.allclose(np.linalg.norm(chunk_vectors, axis=1), 1.0, atol=1e-5), 'Expected normalized embedding vectors.'
print(f'Indexed {len(chunks)} chunks with {MODEL_NAME}; vector dimension: {chunk_vectors.shape[1]}')

## 5. Retrieve evidence: lexical baseline and semantic ranking

Keyword retrieval is useful when a question uses an exact policy term. It is a deliberately simple English-language baseline here. Semantic retrieval is the main Lab 1 method: it ranks chunks by cosine similarity between the E5 query vector and each E5 passage vector.

Try the focused question first, then change it to compare results. A good retrieval result is not a final answer until you read the excerpt.

In [ ]:
QUESTION = 'What actions should schools take to protect learner data when adopting generative AI?'
TOP_K = 5
STOP_WORDS = {'a', 'an', 'and', 'are', 'for', 'how', 'in', 'is', 'of', 'should', 'the', 'to', 'what', 'when', 'with'}

def english_tokens(text: str) -> set[str]:
    return {token for token in re.findall(r'[a-z0-9]+', text.lower()) if token not in STOP_WORDS}

def retrieve_keyword(query: str, limit: int = TOP_K) -> list[RetrievalCandidate]:
    terms = english_tokens(query)
    candidates = [{'method': 'keyword', 'score': len(terms & english_tokens(chunk['text'])) / max(1, len(terms)), 'chunk': chunk} for chunk in chunks]
    return sorted(candidates, key=lambda item: item['score'], reverse=True)[:limit]

def retrieve_semantic(query: str, limit: int = TOP_K) -> tuple[np.ndarray, list[RetrievalCandidate]]:
    query_vector = embedding_model.encode([f'query: {query}'], normalize_embeddings=True, convert_to_numpy=True)[0]
    scores = chunk_vectors @ query_vector  # normalized dot product == cosine similarity
    best_indices = np.argsort(scores)[::-1][:limit]
    return query_vector, [{'method': 'semantic-e5', 'score': float(scores[index]), 'chunk': chunks[index]} for index in best_indices]

query_vector, semantic_results = retrieve_semantic(QUESTION)
keyword_results = retrieve_keyword(QUESTION)
for name, results in [('Semantic E5', semantic_results), ('Keyword baseline', keyword_results[:3])]:
    print(f'\n{name} top results:')
    for result in results:
        print(f"{result['score']:.3f}  {result['chunk']['chunk_id']}  page {result['chunk']['page_start']}")
        print(' ', result['chunk']['text'][:220], '...')

first_index = chunks.index(semantic_results[0]['chunk'])
manual_cosine = float(np.dot(query_vector, chunk_vectors[first_index]))
assert math.isclose(manual_cosine, semantic_results[0]['score'], abs_tol=1e-6)
print(f"\nVerified: top score {semantic_results[0]['score']:.6f} equals its normalized-vector dot product {manual_cosine:.6f}.")

## 6. Inspect the embedding landscape and cosine scores

The t-SNE chart projects the high-dimensional embedding space into two dimensions. Nearby points can help you explore local neighborhoods, but the map is **not** used to rank evidence. Retrieval continues to use full-dimensional cosine similarity, shown separately in the bar chart.

In [ ]:
all_vectors = np.vstack([chunk_vectors, query_vector])
perplexity = min(30, max(2, (len(all_vectors) - 1) // 3))
embedding_map = TSNE(n_components=2, metric='cosine', init='pca', learning_rate='auto', perplexity=perplexity, random_state=42).fit_transform(all_vectors)

top_chunk_ids = {result['chunk']['chunk_id'] for result in semantic_results}
page_bands = np.array([(chunk['page_start'] - 1) // 8 + 1 for chunk in chunks])
top_indices = [index for index, chunk in enumerate(chunks) if chunk['chunk_id'] in top_chunk_ids]
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(embedding_map[:-1, 0], embedding_map[:-1, 1], c=page_bands, cmap='viridis', alpha=0.5, s=28)
ax.scatter(embedding_map[top_indices, 0], embedding_map[top_indices, 1], facecolors='none', edgecolors='#d62728', linewidths=2, s=150, label='Top retrieved chunks')
ax.scatter(embedding_map[-1, 0], embedding_map[-1, 1], marker='*', color='#ff7f0e', edgecolors='black', s=300, label='Query')
for result in semantic_results[:3]:
    index = chunks.index(result['chunk'])
    ax.annotate(result['chunk']['chunk_id'], embedding_map[index], xytext=(5, 5), textcoords='offset points', fontsize=8)
fig.colorbar(scatter, ax=ax, label='Source page band (8 pages each)')
ax.set(title='t-SNE embedding landscape — exploration only, not the ranking rule', xlabel='t-SNE dimension 1', ylabel='t-SNE dimension 2')
ax.legend(loc='best')
plt.show()

MIN_GROUNDED_SCORE = 0.62
labels = [f"p{result['chunk']['page_start']}: {result['chunk']['chunk_id'].split('-')[-1]}" for result in semantic_results[::-1]]
scores = [result['score'] for result in semantic_results[::-1]]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(labels, scores, color='#326b9b')
ax.axvline(MIN_GROUNDED_SCORE, color='#d62728', linestyle='--', label=f'Grounding threshold ({MIN_GROUNDED_SCORE:.2f})')
ax.set(xlim=(0, 1), xlabel='Cosine similarity in the full embedding space', title='Semantic retrieval scores for the current query')
ax.legend(loc='lower right')
plt.show()

## 7. Inspect evidence, then return a bounded response

The response below is extractive and templated: it points to retrieved evidence instead of using a hidden language model. The broad-question rule is intentionally simple and visible, so learners can critique it. In a production system, teams would evaluate thresholds and decision rules with representative questions and human review.

In [ ]:
BROAD_QUESTIONS = {'what does unesco say about generative ai', 'tell me about generative ai', 'what is generative ai'}

def needs_clarification(question: str) -> bool:
    normalized = re.sub(r'[^a-z0-9 ]+', '', question.lower()).strip()
    return normalized in BROAD_QUESTIONS or len(english_tokens(question)) < 3

def citation_from(candidate: RetrievalCandidate) -> Citation:
    chunk = candidate['chunk']
    return {'title': chunk['title'], 'chunk_id': chunk['chunk_id'], 'source_page': f"p. {chunk['page_start']}", 'source_url': chunk['source_url'], 'score': round(candidate['score'], 3)}

def answer_from_evidence(question: str) -> SafeAnswer:
    if needs_clarification(question):
        return {'status': 'ambiguous', 'answer': 'Please ask a focused question about a policy action, learner group, or risk in the approved UNESCO guidance.', 'citations': []}
    _, candidates = retrieve_semantic(question, limit=TOP_K)
    best = candidates[0]
    if best['score'] < MIN_GROUNDED_SCORE:
        return {'status': 'not_found', 'answer': 'This approved UNESCO corpus does not provide sufficiently strong evidence for that question.', 'citations': []}
    excerpt = best['chunk']['text'][:520].rsplit(' ', 1)[0] + '…'
    return {'status': 'grounded', 'answer': f"The retrieved UNESCO evidence says: {excerpt}", 'citations': [citation_from(best)]}

result = answer_from_evidence(QUESTION)
pprint(result)
assert result['status'] == 'grounded'
assert result['citations'] and result['citations'][0]['source_url'] == SOURCE_URL
for prompt in [QUESTION, 'What does UNESCO say about generative AI?', 'What is the capital of France?']:
    print(f"{answer_from_evidence(prompt)['status']:10} | {prompt}")
print('\nCheckpoint: read the top excerpt for the focused prompt. Does it support the claim you would make? If not, revise the question or return a safe response.')

## 8. Hand off to the Lab 1 starter application

You have now built the evidence path behind the rag-v0 starter:

- A ChunkRecord supplies the text, chunk ID, source URL, and page fields of the app's SourceDocument.
- A RetrievalCandidate becomes a displayed Citation.
- grounded, ambiguous, and not_found map directly to AnswerResult.status.

The next Lab 1 task is to implement this same transparent pipeline in starter/rag-v0, retaining its local-corpus boundary and visible citations. Do not copy in a hidden fallback answer: a response should look grounded only after it has retrieved inspectable evidence.

### Beyond Lab 1

Larger systems may add hybrid retrieval, reranking, permissions checks before retrieval, index refreshes, monitoring, and formal evaluation sets. Those features are useful only when they preserve—not obscure—the inspectable evidence path you practised here.